In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

# -------------------------------------------------
# Device selection (ROCm or CPU)
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# -------------------------------------------------
# Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [4]:

# Convert to tensors
X_train_gpu = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_gpu  = torch.tensor(X_test,  dtype=torch.float32).to(device)

y_train_gpu = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_gpu  = torch.tensor(y_test,  dtype=torch.long).to(device)

In [ ]:
epochs = 300
batch_size = 8
lr = 0.001

# -------------------------------------------------
# Define a simple MLP model
# -------------------------------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -------------------------------------------------
# Loss and optimizer
# -------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# -------------------------------------------------
# Training loop
# -------------------------------------------------

dataset = torch.utils.data.TensorDataset(X_train_gpu, y_train_gpu)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(loader):.4f}")

#


Epoch 1/300 - Loss: 1.1002
Epoch 2/300 - Loss: 1.0646
Epoch 3/300 - Loss: 1.0298
Epoch 4/300 - Loss: 0.9924
Epoch 5/300 - Loss: 0.9491
Epoch 6/300 - Loss: 0.8954
Epoch 7/300 - Loss: 0.8302
Epoch 8/300 - Loss: 0.7594
Epoch 9/300 - Loss: 0.6853
Epoch 10/300 - Loss: 0.6171
Epoch 11/300 - Loss: 0.5588
Epoch 12/300 - Loss: 0.5085
Epoch 13/300 - Loss: 0.4684
Epoch 14/300 - Loss: 0.4343
Epoch 15/300 - Loss: 0.4058
Epoch 16/300 - Loss: 0.3819
Epoch 17/300 - Loss: 0.3604
Epoch 18/300 - Loss: 0.3421
Epoch 19/300 - Loss: 0.3267
Epoch 20/300 - Loss: 0.3113
Epoch 21/300 - Loss: 0.2991
Epoch 22/300 - Loss: 0.2844
Epoch 23/300 - Loss: 0.2718
Epoch 24/300 - Loss: 0.2590
Epoch 25/300 - Loss: 0.2453
Epoch 26/300 - Loss: 0.2361
Epoch 27/300 - Loss: 0.2242
Epoch 28/300 - Loss: 0.2110
Epoch 29/300 - Loss: 0.2006
Epoch 30/300 - Loss: 0.1911
Epoch 31/300 - Loss: 0.1798
Epoch 32/300 - Loss: 0.1706
Epoch 33/300 - Loss: 0.1623
Epoch 34/300 - Loss: 0.1524
Epoch 35/300 - Loss: 0.1448
Epoch 36/300 - Loss: 0.1371
E

In [ ]:
# -------------------------------------------------
# Evaluation
# -------------------------------------------------

model.eval()
with torch.no_grad():
    preds = model(X_test_gpu)
    correct = (preds.argmax(dim=1) == y_test_gpu).sum().item()
    acc = correct / len(y_test)

print(f"Test accuracy: {acc:.4f}")

Test accuracy: 0.9667
